# Controlled ablation: H2 (two-backbone) vs H3 (three-backbone)

**Purpose.** This notebook removes reviewer limitations 5&ndash;11 by turning the two
separate notebooks (`SEED42_skin_cancer_HAM10000.ipynb` and `SEED42_PROPOSED_MODEL.ipynb`)
into one *controlled* experiment where the **only** difference between H2 and H3 is the
presence of the third backbone.

| Arm | Backbones | Training harness | Purpose |
|---|---|---|---|
| **H2-original** | InceptionV3 + DenseNet121 | original (batch 32, no augmentation, fast JPEG decode) | the published baseline, as-is |
| **H2-controlled** | InceptionV3 + DenseNet121 | **identical to H3** | isolates the *harness* effect |
| **H3-controlled** | InceptionV3 + DenseNet121 + ConvNeXt-Large | **identical to H2-controlled** | isolates the *third-backbone* effect |

**The key idea that makes this cheap.** H2-controlled and H3-controlled are *the same
trained weights*. The 2-backbone fusion and the 3-backbone fusion are both computed
post-hoc from the saved per-model probability arrays. There is therefore **no possible
harness drift** between them &mdash; identical batch size, augmentation, optimizer, LR,
epochs, split and seed hold *by construction*, not by promise.

---

## How to use this notebook

**Path A &mdash; you already ran the proposed model (fastest, ~2 minutes).**
If your three proposed-model runs saved `prob_inception.npy`, `prob_densenet.npy`,
`prob_convnext.npy` and `y_true.npy`, then H2-controlled needs **zero new training**.
Fill in `RUN_REGISTRY` below, set `MODE = "analyze"`, and run **Part A only**.

**Path B &mdash; you need to (re)train.**
Set `MODE = "train"`, set `ARM` and `SEED`, run **Part B** once per (arm, seed), then
come back and run **Part A**.

> Run cells top-to-bottom. Part A and Part B are independent; you never need both in one session.


## 1. Configuration

This is the only cell you normally edit.

In [ ]:
# ============================================================================
#  CONFIG  -- controlled ablation
#  Env vars must be set BEFORE tensorflow is first imported. This cell runs first.
# ============================================================================
import os, json

# ---------- what to do this session ----------
# "analyze" -> Part A only: build the ablation table / figures from saved .npy probs.
# "train"   -> Part B: train one ARM at one SEED, save probs, then switch to "analyze".
MODE = "analyze"

# ---------- which arm / seed Part B should train (ignored when MODE="analyze") ----------
# ARM = "controlled"  -> trains InceptionV3, DenseNet121, ConvNeXt-Large under the H3 harness
# ARM = "h2_original" -> trains InceptionV3, DenseNet121 under the ORIGINAL H2 harness
ARM  = "controlled"
SEED = 1000

# ---------- the seeds reported in Table III ----------
SEEDS = [42, 1000, 50000]

# ---------- LIMITATION 11: figures must come from a run that is IN Table III ----------
# Every figure (confusion matrix, ROC, per-class table) is generated from this seed's
# saved probabilities, and the seed is stamped into the figure title + filename.
FIGURE_SEED = 42

# ============================================================================
#  CONTROLLED HARNESS  -- LIMITATIONS 6, 7, 8, 9
#  These values are shared by H2-controlled and H3-controlled. Because both arms are
#  computed from ONE set of trained weights, they cannot drift apart.
# ============================================================================
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 16      # LIMITATION 6: identical for H2 and H3 (16 chosen: ConvNeXt-L OOMs at 32)
LR          = 1e-4    # LIMITATION 8: identical
EPOCHS      = 50      # LIMITATION 8: identical
VAL_SPLIT   = 0.10    # LIMITATION 8: identical (stratified, same SEED -> same rows)
NUM_CLASSES = 6

# LIMITATION 7: identical augmentation for H2 and H3
USE_AUGMENTATION = True
NOISE_STDDEV     = 10.0    # Gaussian noise SD, on the 0-255 pixel scale (see FIX note below)
AUG_HFLIP        = True
AUG_ROTATION     = 0.0

USE_CLASS_WEIGHTS = True
FINE_TUNE_BASE          = True    # InceptionV3 / DenseNet121: full fine-tune
CONVNEXT_FINE_TUNE_BASE = False   # ConvNeXt-Large: frozen base (declared, not silent)

INCEPTION_FC_UNITS    = 512
DENSENET_CONV_FILTERS = 512
DENSENET_FC1_UNITS    = 1024
DENSENET_FC2_UNITS    = 512
CONVNEXT_FC_UNITS     = 512

# ---------- ORIGINAL H2 harness (only used when ARM="h2_original") ----------
H2_ORIG_BATCH_SIZE       = 32
H2_ORIG_USE_AUGMENTATION = False
H2_ORIG_ACCURATE_DECODE  = False   # original used tf.io.decode_image defaults

# ============================================================================
#  FUSION WEIGHTS  -- LIMITATION 9
# ============================================================================
# H3 weights, set a-priori (NOT tuned on Derm7pt):
W_INCEPTION_H3 = 0.25
W_DENSENET_H3  = 0.30
W_CONVNEXT_H3  = 0.45

# H2-controlled = the SAME two weights, renormalised to sum to 1 after dropping ConvNeXt.
# 0.25/0.55 = 0.4545 and 0.30/0.55 = 0.5455 -- i.e. almost exactly the original 0.45/0.55,
# so the ablation is not confounded by a change in the InceptionV3:DenseNet121 balance.
_s = W_INCEPTION_H3 + W_DENSENET_H3
W_INCEPTION_H2C = W_INCEPTION_H3 / _s
W_DENSENET_H2C  = W_DENSENET_H3  / _s

# H2-original weights, as published:
W_INCEPTION_H2O = 0.45
W_DENSENET_H2O  = 0.55

# ---------- classes ----------
COMMON_CLASSES = ["MEL", "NV", "BCC", "BKL", "DF", "VASC"]
LABELS = COMMON_CLASSES
CLASS_TO_IDX = {c: i for i, c in enumerate(COMMON_CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

HAM_DX_TO_CLASS = {"mel": "MEL", "nv": "NV", "bcc": "BCC",
                   "bkl": "BKL", "df": "DF", "vasc": "VASC"}
DERM7PT_DIAG_TO_CLASS = [
    ("melanoma",             "MEL"),
    ("nevus",                "NV"),
    ("basal cell carcinoma", "BCC"),
    ("seborrheic keratosis", "BKL"),
    ("dermatofibroma",       "DF"),
    ("vascular",             "VASC"),
]
DERM7PT_IMAGE_COLUMN    = "derm"
ENFORCE_DERM7PT_COUNTS  = True
EXPECTED_DERM7PT_COUNTS = {"NV": 575, "MEL": 252, "BKL": 45,
                           "BCC": 42, "VASC": 29, "DF": 20}   # total 963

# ---------- reproducibility env (must precede `import tensorflow`) ----------
DETERMINISTIC = True
FRESH_START   = True
os.environ["PYTHONHASHSEED"]          = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"]    = "1"
os.environ["TF_CUDNN_DETERMINISTIC"]  = "1"
os.environ["TF_CUDNN_USE_AUTOTUNE"]   = "0"
os.environ["TF_ENABLE_ONEDNN_OPTS"]   = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print(f"MODE={MODE}  ARM={ARM}  SEED={SEED}")
print(f"H2-controlled fusion weights: Inception={W_INCEPTION_H2C:.4f}  DenseNet={W_DENSENET_H2C:.4f}")
print(f"H3-controlled fusion weights: Inception={W_INCEPTION_H3:.2f}  DenseNet={W_DENSENET_H3:.2f}  ConvNeXt={W_CONVNEXT_H3:.2f}")


MODE=analyze  ARM=controlled  SEED=1000
H2-controlled fusion weights: Inception=0.4545  DenseNet=0.5455
H3-controlled fusion weights: Inception=0.25  DenseNet=0.30  ConvNeXt=0.45


## 2. Colab setup

Detects Colab, optionally mounts Drive (**strongly recommended** &mdash; a full training arm
does not finish in one Colab session), and creates the run directory layout:

```
BASE_DIR/
  runs/
    h2_original/seed42/{checkpoints, results}/
    controlled/seed42/{checkpoints, results}/
    ...
  ablation/          <- tables + figures produced by Part A
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ---------------------------------------------------------------------------
# Environment detection + directory layout (Colab / Kaggle / local all work)
# ---------------------------------------------------------------------------
import os, sys, subprocess

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IN_COLAB  = _in_colab()
IN_KAGGLE = os.path.isdir("/kaggle/input") or "KAGGLE_URL_BASE" in os.environ
print("Environment:", "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "local"))

# --- Drive persistence (Colab) --------------------------------------------
# Set False only for a quick throwaway test. Training WILL be interrupted otherwise.
USE_DRIVE = True

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/skin_ablation"
elif IN_KAGGLE:
    BASE_DIR = "/kaggle/working/skin_ablation"
else:
    BASE_DIR = os.path.abspath("./skin_ablation")

RUNS_DIR     = os.path.join(BASE_DIR, "runs")
ABLATION_DIR = os.path.join(BASE_DIR, "ablation")
os.makedirs(RUNS_DIR, exist_ok=True)
os.makedirs(ABLATION_DIR, exist_ok=True)

def run_dir(arm, seed):
    """Canonical location for one (arm, seed) run."""
    d = os.path.join(RUNS_DIR, arm, f"seed{seed}")
    return {"root": d,
            "ckpt": os.path.join(d, "checkpoints"),
            "results": os.path.join(d, "results")}

print("BASE_DIR    :", BASE_DIR)
print("RUNS_DIR    :", RUNS_DIR)
print("ABLATION_DIR:", ABLATION_DIR)

# --- GPU report (batch size 16 assumes >=15 GB for ConvNeXt-Large) ---------
try:
    print("\n" + subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                                 "--format=csv,noheader"],
                                capture_output=True, text=True).stdout.strip())
except Exception:
    print("\nNo nvidia-smi (CPU session?) -- training will be impractically slow.")


Environment: Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR    : /content/drive/MyDrive/skin_ablation
RUNS_DIR    : /content/drive/MyDrive/skin_ablation/runs
ABLATION_DIR: /content/drive/MyDrive/skin_ablation/ablation

NVIDIA A100-SXM4-40GB, 40960 MiB


---
# PART A &mdash; Controlled ablation from saved probabilities

Run this part when the per-model probability arrays already exist. It produces:

* the **ablation table** (H2-original / H2-controlled / H3-controlled) &mdash; *limitation 10*
* **per-seed rows + mean &plusmn; SD** consistent with Table III &mdash; *limitations 5, 11*
* **McNemar** and **paired-bootstrap** tests for H3 vs H2-controlled
* **confusion matrix + ROC** regenerated from the exact seed reported &mdash; *limitation 11*
* LaTeX ready to paste into the IEEE manuscript

## A1. Point the notebook at your existing runs

Edit `RUN_REGISTRY` so each entry is the folder containing `y_true.npy`,
`prob_inception.npy`, `prob_densenet.npy` (and `prob_convnext.npy` for the controlled arm).
Set a value to `None` if you have not run it yet.

In [ ]:
# ---------------------------------------------------------------------------
# A1. Where the saved probability arrays live.
#     Leave as run_dir(...) if Part B produced them; otherwise paste your own paths
#     (e.g. the results/ folders from your three SEED*_PROPOSED_MODEL runs).
# ---------------------------------------------------------------------------
RUN_REGISTRY = {
    # arm            seed -> results folder
    "h2_original": {s: run_dir("h2_original", s)["results"] for s in SEEDS},
    "controlled":  {s: run_dir("controlled",  s)["results"] for s in SEEDS},
}

# --- EXAMPLE of pointing at pre-existing runs on Drive ----------------------
# RUN_REGISTRY["controlled"] = {
#     42:    "/content/drive/MyDrive/proposed/seed42/results",
#     1000:  "/content/drive/MyDrive/proposed/seed1000/results",
#     50000: "/content/drive/MyDrive/proposed/seed50000/results",
# }
# RUN_REGISTRY["h2_original"] = {
#     42:    "/content/drive/MyDrive/baseline/seed42/results",
#     1000:  "/content/drive/MyDrive/baseline/seed1000/results",
#     50000: "/content/drive/MyDrive/baseline/seed50000/results",
# }

REQUIRED = {
    "h2_original": ["y_true.npy", "prob_inception.npy", "prob_densenet.npy"],
    "controlled":  ["y_true.npy", "prob_inception.npy", "prob_densenet.npy", "prob_convnext.npy"],
}

print(f"{'arm':<14}{'seed':<8}{'status':<12}path")
print("-" * 90)
AVAILABLE = {}
for arm, seedmap in RUN_REGISTRY.items():
    for seed, path in seedmap.items():
        if not path or not os.path.isdir(path):
            print(f"{arm:<14}{seed:<8}{'MISSING':<12}{path}")
            continue
        missing = [f for f in REQUIRED[arm] if not os.path.isfile(os.path.join(path, f))]
        if missing:
            print(f"{arm:<14}{seed:<8}{'INCOMPLETE':<12}{path}   (no {', '.join(missing)})")
        else:
            AVAILABLE.setdefault(arm, {})[seed] = path
            print(f"{arm:<14}{seed:<8}{'OK':<12}{path}")

print("\nUsable runs:", {a: sorted(v) for a, v in AVAILABLE.items()})


arm           seed    status      path
------------------------------------------------------------------------------------------
h2_original   42      OK          /content/drive/MyDrive/skin_ablation/runs/h2_original/seed42/results
h2_original   1000    OK          /content/drive/MyDrive/skin_ablation/runs/h2_original/seed1000/results
h2_original   50000   OK          /content/drive/MyDrive/skin_ablation/runs/h2_original/seed50000/results
controlled    42      OK          /content/drive/MyDrive/skin_ablation/runs/controlled/seed42/results
controlled    1000    OK          /content/drive/MyDrive/skin_ablation/runs/controlled/seed1000/results
controlled    50000   OK          /content/drive/MyDrive/skin_ablation/runs/controlled/seed50000/results

Usable runs: {'h2_original': [42, 1000, 50000], 'controlled': [42, 1000, 50000]}


## A2. Metric helpers

Identical metric definitions to both source notebooks (macro-averaged, one-vs-rest AUC),
so the numbers are directly comparable to your existing Table III.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

def overall_metrics(y_true, prob):
    """Macro-averaged metrics, as reported in Table III."""
    y_pred = np.argmax(prob, axis=1)
    yb = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    return {
        "Accuracy":  accuracy_score(y_true, y_pred) * 100,
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Recall":    recall_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "F1":        f1_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "AUC":       roc_auc_score(yb, prob, average="macro", multi_class="ovr") * 100,
    }

def per_class_table(y_true, prob):
    y_pred = np.argmax(prob, axis=1)
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    yb = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
    rows = []
    for i, cls in enumerate(COMMON_CLASSES):
        tp = int(cm[i, i]); fp = int(cm[:, i].sum() - tp)
        fn = int(cm[i, :].sum() - tp); tn = int(cm.sum() - tp - fp - fn)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        try:
            auc = roc_auc_score(yb[:, i], prob[:, i])
        except ValueError:
            auc = float("nan")
        rows.append([cls, tp, fp, fn, tn, round(prec, 4), round(rec, 4),
                     round(f1, 4), round(spec, 4), round(auc, 4)])
    return pd.DataFrame(rows, columns=["Class", "TP", "FP", "FN", "TN",
                                       "Precision", "Recall", "F1", "Specificity", "AUC"])

def load_probs(arm, seed):
    """Return (y_true, dict_of_probability_arrays) for one run."""
    p = AVAILABLE[arm][seed]
    out = {"inception": np.load(os.path.join(p, "prob_inception.npy")),
           "densenet":  np.load(os.path.join(p, "prob_densenet.npy"))}
    cpath = os.path.join(p, "prob_convnext.npy")
    if os.path.isfile(cpath):
        out["convnext"] = np.load(cpath)
    return np.load(os.path.join(p, "y_true.npy")), out

def fuse(probs, weights):
    """Weighted-sum fusion (Eq. 1). weights: {name: w}."""
    keys = list(weights)
    missing = [k for k in keys if k not in probs]
    assert not missing, f"missing probability arrays: {missing}"
    return sum(weights[k] * probs[k] for k in keys)

print("Metric helpers ready.")


Metric helpers ready.


## A3. Integrity guard &mdash; the arms must share one test set

Before anything is compared, this asserts that every run used the **same 963-image
Derm7pt test set in the same order**. If the label vectors differ, the comparison is
invalid and the notebook stops rather than producing a misleading table.

This is the check that makes limitation 11 defensible: figures and tables can only be
"consistent with the runs in Table III" if the underlying test tensor is provably identical.

In [ ]:
# ---------------------------------------------------------------------------
# A3. Verify all runs share one identical, identically-ordered test set.
# ---------------------------------------------------------------------------
ref_y, ref_tag = None, None
for arm, seedmap in AVAILABLE.items():
    for seed in sorted(seedmap):
        y, _ = load_probs(arm, seed)
        tag = f"{arm}/seed{seed}"
        if ref_y is None:
            ref_y, ref_tag = y, tag
            continue
        assert len(y) == len(ref_y), (
            f"TEST-SET SIZE MISMATCH: {tag} has N={len(y)} but {ref_tag} has N={len(ref_y)}. "
            "The arms did not evaluate the same Derm7pt subset -- fix the label mapping "
            "before comparing.")
        assert np.array_equal(y, ref_y), (
            f"TEST-SET ORDER/LABEL MISMATCH between {tag} and {ref_tag}. "
            "Same size but different labels or ordering -- predictions are not aligned.")

y_true = ref_y
counts = pd.Series(y_true).map(IDX_TO_CLASS).value_counts()
print(f"All runs share one test set.  N = {len(y_true)}")
print(counts.to_string())
exp_total = sum(EXPECTED_DERM7PT_COUNTS.values())
if len(y_true) != exp_total:
    print(f"\nNOTE: N={len(y_true)} but CONFIG expects {exp_total}. "
          "Check the Derm7pt label mapping before reporting.")


All runs share one test set.  N = 963
NV      575
MEL     252
BKL      45
BCC      42
VASC     29
DF       20


## A4. The ablation table &mdash; limitations 5, 9, 10

Three arms, per seed, plus mean &plusmn; SD across the seeds reported in Table III.

Read it as a two-step decomposition:

* **H2-original &rarr; H2-controlled** = the effect of the *training harness alone*
  (batch 32&rarr;16, augmentation off&rarr;on, decode). No architecture change.
* **H2-controlled &rarr; H3-controlled** = the effect of *adding ConvNeXt-Large alone*.
  Same weights, same everything &mdash; the isolated contribution of the third backbone.

Reporting both rows is what converts "our model is better" into "the third backbone
contributes X points, and the harness contributes Y".

In [ ]:
# ---------------------------------------------------------------------------
# A4. Build the ablation table
# ---------------------------------------------------------------------------
ARM_SPECS = [
    # (display name, source arm, fusion weights)
    ("H2-original   (InceptionV3 + DenseNet121)",
     "h2_original", {"inception": W_INCEPTION_H2O, "densenet": W_DENSENET_H2O}),
    ("H2-controlled (InceptionV3 + DenseNet121)",
     "controlled",  {"inception": W_INCEPTION_H2C, "densenet": W_DENSENET_H2C}),
    ("H3-controlled (+ ConvNeXt-Large)",
     "controlled",  {"inception": W_INCEPTION_H3, "densenet": W_DENSENET_H3,
                     "convnext":  W_CONVNEXT_H3}),
]

per_seed_rows, fused_cache = [], {}
for disp, arm, weights in ARM_SPECS:
    for seed in SEEDS:
        if arm not in AVAILABLE or seed not in AVAILABLE[arm]:
            continue
        _, probs = load_probs(arm, seed)
        if any(k not in probs for k in weights):
            continue
        fp = fuse(probs, weights)
        fused_cache[(disp, seed)] = fp
        per_seed_rows.append({"Model": disp, "Seed": seed, **overall_metrics(y_true, fp)})

per_seed = pd.DataFrame(per_seed_rows)
if per_seed.empty:
    raise SystemExit("No runs available. Fill in RUN_REGISTRY (A1) or run Part B first.")

METRICS = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

print("=" * 100)
print("PER-SEED RESULTS  (must match the individual runs behind Table III)")
print("=" * 100)
print(per_seed.round(2).to_string(index=False))

# --- mean +/- sample SD (ddof=1), the convention used in Table III ---------
agg = per_seed.groupby("Model")[METRICS].agg(["mean", lambda s: s.std(ddof=1)])
agg.columns = [f"{m}_{'mean' if k=='mean' else 'sd'}" for m, k in agg.columns]
order = [d for d, _, _ in ARM_SPECS if d in agg.index]
agg = agg.loc[order]

ablation = pd.DataFrame(index=agg.index)
for m in METRICS:
    ablation[m] = [f"{r[f'{m}_mean']:.2f} \u00b1 {r[f'{m}_sd']:.2f}"
                   if pd.notna(r[f'{m}_sd']) else f"{r[f'{m}_mean']:.2f}"
                   for _, r in agg.iterrows()]
ablation.insert(0, "n seeds", per_seed.groupby("Model").size().loc[order].values)

print("\n" + "=" * 100)
print(f"ABLATION TABLE  (mean \u00b1 SD over seeds {SEEDS})")
print("=" * 100)
print(ablation.to_string())

# --- the two isolated effects ---------------------------------------------
def _delta(a, b, label):
    if a not in agg.index or b not in agg.index:
        return None
    d = {m: agg.loc[b, f"{m}_mean"] - agg.loc[a, f"{m}_mean"] for m in METRICS}
    return pd.Series(d, name=label)

deltas = [d for d in [
    _delta(ARM_SPECS[0][0], ARM_SPECS[1][0], "Harness effect  (H2-orig -> H2-ctrl)"),
    _delta(ARM_SPECS[1][0], ARM_SPECS[2][0], "3rd backbone    (H2-ctrl -> H3-ctrl)"),
] if d is not None]

if deltas:
    print("\n" + "=" * 100)
    print("ISOLATED EFFECTS (percentage points, mean across seeds)")
    print("=" * 100)
    print(pd.DataFrame(deltas).round(2).to_string())

per_seed.round(4).to_csv(os.path.join(ABLATION_DIR, "ablation_per_seed.csv"), index=False)
ablation.to_csv(os.path.join(ABLATION_DIR, "ablation_table.csv"))
agg.round(4).to_csv(os.path.join(ABLATION_DIR, "ablation_mean_sd.csv"))
print(f"\nSaved -> {ABLATION_DIR}/ablation_{{per_seed,table,mean_sd}}.csv")


PER-SEED RESULTS  (must match the individual runs behind Table III)
                                    Model  Seed  Accuracy  Precision  Recall    F1   AUC
H2-original   (InceptionV3 + DenseNet121)    42     72.07      58.43   52.74 53.12 89.02
H2-original   (InceptionV3 + DenseNet121)  1000     69.99      67.02   44.66 49.25 88.89
H2-original   (InceptionV3 + DenseNet121) 50000     71.03      70.69   47.66 51.22 88.47
H2-controlled (InceptionV3 + DenseNet121)    42     70.20      60.91   51.35 51.51 88.84
H2-controlled (InceptionV3 + DenseNet121)  1000     69.16      66.68   50.99 52.28 88.49
H2-controlled (InceptionV3 + DenseNet121) 50000     68.12      63.75   47.54 48.11 86.07
         H3-controlled (+ ConvNeXt-Large)    42     73.73      67.46   55.27 56.90 90.51
         H3-controlled (+ ConvNeXt-Large)  1000     73.31      71.68   54.32 57.24 90.35
         H3-controlled (+ ConvNeXt-Large) 50000     73.42      67.72   56.46 56.54 89.74

ABLATION TABLE  (mean ± SD over seeds [42

## A5. Is the H3 gain statistically real?

Three seeds is too few for a credible paired *t*-test, so this cell reports two tests
that a reviewer will accept:

**McNemar's test** (per seed) &mdash; the correct paired test when two classifiers are
evaluated on *the same* test items. It looks only at the images where the two models
disagree: `b` = H2-controlled right / H3-controlled wrong, `c` = the reverse. Under the
null, disagreements split 50/50, so an exact binomial test on `b` vs `c` gives the
*p*-value. This is far more powerful than comparing two accuracy numbers.

**Paired bootstrap** &mdash; resamples the 963 test images (with replacement) and
recomputes the H3&minus;H2 macro-F1 difference on each resample, giving a 95% CI on the
gain. Because both models score the same resampled images, per-image difficulty cancels.

Both tests are *within-seed*: they quantify the third backbone's effect given one trained
pair, which is exactly the ablation claim.

In [ ]:
# ---------------------------------------------------------------------------
# A5. McNemar (exact) + paired bootstrap: H2-controlled vs H3-controlled
# ---------------------------------------------------------------------------
from scipy.stats import binomtest

NAME_H2C = ARM_SPECS[1][0]
NAME_H3C = ARM_SPECS[2][0]

def mcnemar_exact(y, pa, pb):
    """Exact McNemar. pa = baseline probs, pb = challenger probs."""
    ca = (np.argmax(pa, 1) == y)
    cb = (np.argmax(pb, 1) == y)
    b = int(np.sum(ca & ~cb))   # baseline right, challenger wrong
    c = int(np.sum(~ca & cb))   # baseline wrong, challenger right
    p = binomtest(c, b + c, 0.5).pvalue if (b + c) else 1.0
    return b, c, p

def paired_bootstrap(y, pa, pb, metric="F1", n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(y)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y[idx])) < 2:
            diffs[i] = np.nan
            continue
        diffs[i] = (overall_metrics(y[idx], pb[idx])[metric]
                    - overall_metrics(y[idx], pa[idx])[metric])
    diffs = diffs[~np.isnan(diffs)]
    return diffs.mean(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

rows = []
for seed in SEEDS:
    ka, kb = (NAME_H2C, seed), (NAME_H3C, seed)
    if ka not in fused_cache or kb not in fused_cache:
        continue
    pa, pb = fused_cache[ka], fused_cache[kb]
    b, c, p = mcnemar_exact(y_true, pa, pb)
    acc_d = overall_metrics(y_true, pb)["Accuracy"] - overall_metrics(y_true, pa)["Accuracy"]
    f1_d, lo, hi = paired_bootstrap(y_true, pa, pb, "F1", n_boot=2000, seed=seed)
    rows.append({"Seed": seed,
                 "H2c right / H3c wrong (b)": b,
                 "H2c wrong / H3c right (c)": c,
                 "McNemar p": p,
                 "d Accuracy (pp)": acc_d,
                 "d macro-F1 (pp)": f1_d,
                 "95% CI low": lo, "95% CI high": hi})

if rows:
    sig = pd.DataFrame(rows)
    print("=" * 105)
    print("PAIRED SIGNIFICANCE: H3-controlled vs H2-controlled (same test images, same weights)")
    print("=" * 105)
    print(sig.round(4).to_string(index=False))
    print("\nReading it: c > b means the third backbone fixed more images than it broke.")
    print("A 95% CI that excludes 0 supports a real gain; one that straddles 0 does not.")
    sig.round(6).to_csv(os.path.join(ABLATION_DIR, "significance_h2c_vs_h3c.csv"), index=False)
else:
    print("Need both H2-controlled and H3-controlled for at least one seed.")


KeyboardInterrupt: 

## A6. Figures traceable to a reported run &mdash; limitation 11

Every figure below is regenerated from `FIGURE_SEED`'s **saved probability arrays** &mdash;
the same arrays that produced that seed's row in the per-seed table above. Nothing is
recomputed from a different training run, and the seed is written into both the figure
title and the filename, so provenance survives being dropped into the manuscript.

Caption template for the paper:

> Fig. N. Confusion matrix for the proposed three-backbone model on the Derm7pt external
> test set (*n* = 963), seed 42. Table III reports the mean &plusmn; SD across seeds
> {42, 1000, 50000}; this figure shows the seed-42 run from that set.

In [ ]:
# ---------------------------------------------------------------------------
# A6. Confusion matrix + ROC, stamped with the seed they came from
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

FIG_MODEL = NAME_H3C          # change to NAME_H2C to produce the baseline figures
key = (FIG_MODEL, FIGURE_SEED)
assert key in fused_cache, (
    f"No fused predictions for {FIG_MODEL} at seed {FIGURE_SEED}. "
    f"Available: {sorted(k[1] for k in fused_cache if k[0] == FIG_MODEL)}")
prob_fig = fused_cache[key]
short = "H3ctrl" if FIG_MODEL == NAME_H3C else "H2ctrl"

# --- confusion matrix ------------------------------------------------------
y_pred = np.argmax(prob_fig, axis=1)
cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=COMMON_CLASSES).plot(
    cmap="Blues", ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title(f"Confusion matrix - {FIG_MODEL}\nDerm7pt external test (n={len(y_true)}), seed {FIGURE_SEED}")
plt.tight_layout()
cm_path = os.path.join(ABLATION_DIR, f"confusion_matrix_{short}_seed{FIGURE_SEED}.png")
plt.savefig(cm_path, dpi=300); plt.show()

# --- one-vs-rest ROC -------------------------------------------------------
yb = label_binarize(y_true, classes=np.arange(NUM_CLASSES))
plt.figure(figsize=(6.5, 6))
for i, cls in enumerate(COMMON_CLASSES):
    if yb[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(yb[:, i], prob_fig[:, i])
    plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc_score(yb[:, i], prob_fig[:, i]):.3f})")
macro_auc = roc_auc_score(yb, prob_fig, average="macro", multi_class="ovr")
plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title(f"One-vs-rest ROC - {FIG_MODEL}\nDerm7pt, seed {FIGURE_SEED} (macro AUC={macro_auc:.3f})")
plt.legend(loc="lower right", fontsize=8); plt.tight_layout()
roc_path = os.path.join(ABLATION_DIR, f"roc_{short}_seed{FIGURE_SEED}.png")
plt.savefig(roc_path, dpi=300); plt.show()

# --- per-class table for the SAME run (Table IV) ---------------------------
pc = per_class_table(y_true, prob_fig)
pc.to_csv(os.path.join(ABLATION_DIR, f"per_class_{short}_seed{FIGURE_SEED}.csv"), index=False)
print(f"\nPer-class metrics - {FIG_MODEL}, seed {FIGURE_SEED} (this is your Table IV):")
print(pc.to_string(index=False))

# --- provenance stamp ------------------------------------------------------
prov = {
    "figure_model": FIG_MODEL,
    "figure_seed": FIGURE_SEED,
    "source_results_dir": AVAILABLE["controlled"][FIGURE_SEED],
    "test_set_n": int(len(y_true)),
    "table_iii_seeds": SEEDS,
    "fusion_weights": {"inception": W_INCEPTION_H3, "densenet": W_DENSENET_H3,
                       "convnext": W_CONVNEXT_H3},
    "files": {"confusion_matrix": cm_path, "roc": roc_path},
}
with open(os.path.join(ABLATION_DIR, "figure_provenance.json"), "w") as f:
    json.dump(prov, f, indent=2)
print("\nProvenance written -> ablation/figure_provenance.json")


## A7. LaTeX export (IEEE two-column, `\small` to fit)

Paste straight into the manuscript. Uses `booktabs`; swap to plain `\hline` if you'd
rather not add the package.

In [ ]:
# ---------------------------------------------------------------------------
# A7. IEEE-ready LaTeX for the ablation table
# ---------------------------------------------------------------------------
def latex_ablation(df):
    L = []
    L.append(r"\begin{table}[t]")
    L.append(r"\caption{Controlled ablation on the Derm7pt external test set "
             r"($n=" + str(len(y_true)) + r"$). All arms share batch size, augmentation, "
             r"optimizer, learning rate, epochs, train/validation split and seeds; "
             r"H2-controlled and H3-controlled additionally share the same trained "
             r"InceptionV3 and DenseNet121 weights. Mean $\pm$ SD over seeds "
             r"$\{" + ", ".join(map(str, SEEDS)) + r"\}$.}")
    L.append(r"\label{tab:ablation}")
    L.append(r"\centering\small")
    L.append(r"\begin{tabular}{lccccc}")
    L.append(r"\toprule")
    L.append(r"Configuration & Acc. & Prec. & Rec. & F1 & AUC \\")
    L.append(r"\midrule")
    for name, row in df.iterrows():
        clean = (name.split("(")[0].strip()
                 .replace("H2-original", "H2-original (2 backbones)")
                 .replace("H2-controlled", "H2-controlled (2 backbones)")
                 .replace("H3-controlled", "H3-controlled (3 backbones)"))
        cells = " & ".join(str(row[m]).replace("\u00b1", r"$\pm$") for m in METRICS)
        L.append(f"{clean} & {cells} \\\\")
    L.append(r"\bottomrule")
    L.append(r"\end{tabular}")
    L.append(r"\end{table}")
    return "\n".join(L)

tex = latex_ablation(ablation)
with open(os.path.join(ABLATION_DIR, "ablation_table.tex"), "w") as f:
    f.write(tex)
print(tex)
print(f"\nSaved -> {ABLATION_DIR}/ablation_table.tex")


---
# PART B &mdash; Controlled training

Only needed when the probability arrays for an arm/seed do not yet exist.

Set `MODE = "train"`, choose `ARM` and `SEED` in the CONFIG cell, then run this part.
Repeat once per (arm, seed). Everything writes into `runs/<arm>/seed<N>/`, so the arms
can never overwrite each other.

**Rough cost per seed on a Colab T4** (HAM10000, ~8.2k training images, 50 epochs):
InceptionV3 ~3 h, DenseNet121 ~3 h, ConvNeXt-Large (frozen base) ~6&ndash;7 h.
Use an A100/L4 runtime if you have one, keep Drive mounted, and re-run the training cell
after a disconnect &mdash; it resumes from the last epoch.

### One fix applied here

The proposed-model notebook wrote `layers.GaussianNoise(NOISE_STDDEV / 255.0)` but applied
it **before** `Rescaling(1/255)`, i.e. to data still on the 0&ndash;255 scale. The effective
noise SD was therefore 10/255 &asymp; 0.039 on a 0&ndash;255 range &mdash; about 0.015% of the
dynamic range, which is numerically no augmentation at all. This notebook applies
`GaussianNoise(NOISE_STDDEV)` on the 0&ndash;255 scale as the comment intended.

> Because this changes the augmentation, any arm you train here is **not** interchangeable
> with your previous proposed-model runs. Either (a) keep the old behaviour by setting
> `NOISE_ON_255_SCALE = False` below so new runs match your existing ones, or (b) retrain
> **all** arms with the fix. Do not mix.

In [ ]:
# ---------------------------------------------------------------------------
# B0. Guard + resolve the harness for the selected ARM
# ---------------------------------------------------------------------------
if MODE != "train":
    print("MODE is not 'train' -- skip the rest of Part B and use Part A.")

# True  -> GaussianNoise(NOISE_STDDEV) on the 0-255 scale  (the fix; noise is real)
# False -> GaussianNoise(NOISE_STDDEV/255) on the 0-255 scale (reproduces the ORIGINAL
#          proposed-model runs, where the noise was effectively zero)
NOISE_ON_255_SCALE = False

assert ARM in ("controlled", "h2_original"), f"unknown ARM: {ARM}"

if ARM == "controlled":
    ACTIVE_BATCH   = BATCH_SIZE
    ACTIVE_AUG     = USE_AUGMENTATION
    ACCURATE_DECODE = True
    BACKBONES      = ["inceptionv3", "densenet121", "convnextlarge"]
else:
    ACTIVE_BATCH   = H2_ORIG_BATCH_SIZE
    ACTIVE_AUG     = H2_ORIG_USE_AUGMENTATION
    ACCURATE_DECODE = H2_ORIG_ACCURATE_DECODE
    BACKBONES      = ["inceptionv3", "densenet121"]

D = run_dir(ARM, SEED)
CHECKPOINT_DIR, RESULTS_DIR = D["ckpt"], D["results"]
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"ARM={ARM}  SEED={SEED}")
print(f"  batch={ACTIVE_BATCH}  augmentation={ACTIVE_AUG}  accurate_decode={ACCURATE_DECODE}")
print(f"  backbones={BACKBONES}")
print(f"  -> {D['root']}")


In [ ]:
# ---------------------------------------------------------------------------
# B1. Download the datasets (kagglehub; works on Colab out of the box)
# ---------------------------------------------------------------------------
try:
    import kagglehub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
    import kagglehub

ham_path  = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
derm_path = kagglehub.dataset_download("menakamohanakumar/derm7pt")
print("HAM10000 ->", ham_path)
print("Derm7pt  ->", derm_path)

HAM10000_HINT, DERM7PT_HINT = ham_path, derm_path


In [ ]:
# ---------------------------------------------------------------------------
# B2. Build train_df / test_df  (identical mapping logic for BOTH arms, so the
#     test set is provably the same 963 images in the same order)
# ---------------------------------------------------------------------------
import pandas as pd

SEARCH_ROOTS = [HAM10000_HINT, DERM7PT_HINT, "/kaggle/input", "."]

def _find_file(filename, roots):
    matches = []
    for root in roots:
        if not root or not os.path.exists(root):
            continue
        for dp, dns, files in os.walk(root):
            dns.sort()
            if filename in files:
                matches.append(os.path.join(dp, filename))
    matches = sorted(set(matches))
    if len(matches) > 1:
        print(f"WARNING: {len(matches)} files named '{filename}'; using the first:\n   "
              + "\n   ".join(matches))
    return matches[0] if matches else None

def _find_meta(filename, hint):
    if hint and os.path.isdir(hint):
        hit = _find_file(filename, [hint])
        if hit:
            return hit
        print(f"NOTE: '{filename}' not under {hint}; widening search.")
    return _find_file(filename, SEARCH_ROOTS)

# ---- HAM10000 (train) ----
ham_meta_path = _find_meta("HAM10000_metadata.csv", HAM10000_HINT)
assert ham_meta_path, "HAM10000_metadata.csv not found"
ham_root = os.path.dirname(ham_meta_path)
ham = pd.read_csv(ham_meta_path)

ham_img_index = {}
for dp, dns, files in os.walk(ham_root):
    dns.sort()
    for f in sorted(files):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            ham_img_index[os.path.splitext(f)[0]] = os.path.join(dp, f)
print("HAM10000 images found:", len(ham_img_index))

ham["path"]      = ham["image_id"].map(ham_img_index)
ham["class_str"] = ham["dx"].map(HAM_DX_TO_CLASS)
ham["label"]     = ham["class_str"].map(CLASS_TO_IDX)
print(f"Dropped {(ham['dx'] == 'akiec').sum()} AKIEC images (absent from Derm7pt).")

train_df = (ham.dropna(subset=["path", "label"])[["path", "label"]]
              .astype({"label": int}).reset_index(drop=True))
print("\nHAM10000 (train) class counts:")
print(train_df["label"].map(IDX_TO_CLASS).value_counts().to_string())

# ---- Derm7pt (external test) ----
derm_meta_path = _find_meta("meta.csv", DERM7PT_HINT)
assert derm_meta_path, "Derm7pt meta.csv not found"
derm_meta_dir = os.path.dirname(derm_meta_path)
derm = pd.read_csv(derm_meta_path)
assert {"diagnosis", DERM7PT_IMAGE_COLUMN} <= set(derm.columns), list(derm.columns)

derm_release = os.path.dirname(derm_meta_dir)
derm_img_dir = next((c for c in [os.path.join(derm_release, "images"),
                                 derm_release, derm_meta_dir] if os.path.isdir(c)),
                    derm_meta_dir)
derm_basename_index = {}
for dp, dns, files in os.walk(derm_img_dir):
    dns.sort()
    for f in sorted(files):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            derm_basename_index.setdefault(os.path.basename(f), os.path.join(dp, f))
print("\nDerm7pt images found:", len(derm_basename_index))

def resolve_derm_path(rel):
    direct = os.path.join(derm_img_dir, str(rel))
    return direct if os.path.isfile(direct) else derm_basename_index.get(os.path.basename(str(rel)))

def derm_to_class(diag):
    d = str(diag).lower()
    for key, cls in DERM7PT_DIAG_TO_CLASS:
        if key in d:
            return cls
    return None

derm["path"]      = derm[DERM7PT_IMAGE_COLUMN].map(resolve_derm_path)
derm["class_str"] = derm["diagnosis"].map(derm_to_class)
derm["label"]     = derm["class_str"].map(CLASS_TO_IDX)

unmapped = derm["class_str"].isna()
if unmapped.sum():
    print(f"WARNING: {unmapped.sum()} Derm7pt rows unmapped (dropped): "
          f"{sorted(derm.loc[unmapped, 'diagnosis'].astype(str).unique())}")

test_df = (derm.dropna(subset=["path", "label"])[["path", "label"]]
              .astype({"label": int}).reset_index(drop=True))
print("\nDerm7pt (test) class counts:")
print(test_df["label"].map(IDX_TO_CLASS).value_counts().to_string())

actual = test_df["label"].map(IDX_TO_CLASS).value_counts().to_dict()
if ENFORCE_DERM7PT_COUNTS:
    for cls, exp in EXPECTED_DERM7PT_COUNTS.items():
        assert actual.get(cls, 0) == exp, (
            f"Derm7pt {cls}: expected {exp}, got {actual.get(cls, 0)}. "
            "Wrong mapping or a different release -- fix before comparing arms.")
    assert len(test_df) == sum(EXPECTED_DERM7PT_COUNTS.values())
    print(f"\nDerm7pt guard passed: N={len(test_df)}")


In [ ]:
# ---------------------------------------------------------------------------
# B3. tf.data pipeline. Decode method follows the ARM so H2-original is faithful.
# ---------------------------------------------------------------------------
import random
import numpy as np
import tensorflow as tf
import sklearn
from sklearn.model_selection import train_test_split

random.seed(SEED); np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
if DETERMINISTIC:
    tf.config.experimental.enable_op_determinism()

print("TF", tf.__version__, "| Keras", tf.keras.__version__,
      "| NumPy", np.__version__, "| sklearn", sklearn.__version__)

try:
    with open(os.path.join(RESULTS_DIR, "requirements.txt"), "w") as _f:
        subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=_f, check=True)
except Exception as e:
    print("Could not freeze env:", e)

AUTOTUNE = tf.data.AUTOTUNE
class_names = LABELS

def _decode(path, label):
    raw = tf.io.read_file(path)
    if ACCURATE_DECODE:
        img = tf.cond(tf.image.is_jpeg(raw),
                      lambda: tf.io.decode_jpeg(raw, channels=3, dct_method="INTEGER_ACCURATE"),
                      lambda: tf.io.decode_png(raw, channels=3))
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, IMG_SIZE, method="bilinear", antialias=False)
    else:
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(img, tf.float32), label     # stays 0-255

_DET_OPTS = tf.data.Options()
_DET_OPTS.experimental_deterministic = True

def make_ds(df, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["path"].values, df["label"].values.astype("int32"))).with_options(_DET_OPTS)
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=False)
    return ds.map(_decode, num_parallel_calls=AUTOTUNE,
                  deterministic=True).batch(ACTIVE_BATCH)

# LIMITATION 8: same SEED + same VAL_SPLIT -> byte-identical split across arms
tr_df, va_df = train_test_split(train_df, test_size=VAL_SPLIT,
                                random_state=SEED, stratify=train_df["label"])
tr_df.to_csv(os.path.join(RESULTS_DIR, "train_split.csv"), index=False)
va_df.to_csv(os.path.join(RESULTS_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(RESULTS_DIR, "test_derm7pt.csv"), index=False)

train_ds = make_ds(tr_df, shuffle=True).prefetch(AUTOTUNE)
val_ds   = make_ds(va_df, shuffle=False).cache().prefetch(AUTOTUNE)
test_ds  = make_ds(test_df, shuffle=False).cache().prefetch(AUTOTUNE)   # never shuffled

print(f"train={len(tr_df)}  val={len(va_df)}  test={len(test_df)}  batch={ACTIVE_BATCH}")

CLASS_WEIGHT = None
if USE_CLASS_WEIGHTS:
    from sklearn.utils.class_weight import compute_class_weight
    cls = np.arange(NUM_CLASSES)
    w = compute_class_weight("balanced", classes=cls, y=tr_df["label"].values)
    CLASS_WEIGHT = {int(c): float(wi) for c, wi in zip(cls, w)}
    print("Class weights:", {IDX_TO_CLASS[k]: round(v, 3) for k, v in CLASS_WEIGHT.items()})

# --- split fingerprint: proves two arms used the same rows --------------------
import hashlib
split_hash = hashlib.sha256(
    ("|".join(sorted(os.path.basename(p) for p in tr_df["path"]))).encode()).hexdigest()[:16]
test_hash = hashlib.sha256(
    ("|".join(os.path.basename(p) for p in test_df["path"])).encode()).hexdigest()[:16]
print(f"\nTRAIN split fingerprint : {split_hash}")
print(f"TEST  order fingerprint : {test_hash}")
print("These must be identical across arms for the ablation to be controlled.")


In [ ]:
# ---------------------------------------------------------------------------
# B4. Model builders. Augmentation is shared by every backbone in the arm
#     (LIMITATION 7) and is a no-op at inference, so Derm7pt is never perturbed.
# ---------------------------------------------------------------------------
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3, DenseNet121, ConvNeXtLarge

def augmentation_block():
    augs = []
    if AUG_HFLIP:
        augs.append(layers.RandomFlip("horizontal", seed=SEED))
    if AUG_ROTATION and AUG_ROTATION > 0:
        augs.append(layers.RandomRotation(AUG_ROTATION, seed=SEED))
    if NOISE_STDDEV and NOISE_STDDEV > 0:
        sd = NOISE_STDDEV if NOISE_ON_255_SCALE else NOISE_STDDEV / 255.0
        augs.append(layers.GaussianNoise(sd))
    return models.Sequential(augs, name="augment") if augs else None

def _maybe_augment(x):
    if ACTIVE_AUG:
        blk = augmentation_block()
        if blk is not None:
            x = blk(x)
    return x

def build_inceptionv3():
    base = InceptionV3(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = FINE_TUNE_BASE
    inp = layers.Input(shape=IMG_SIZE + (3,))
    x = _maybe_augment(inp)
    x = layers.Rescaling(1.0 / 255)(x)
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(INCEPTION_FC_UNITS, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return models.Model(inp, out, name="InceptionV3_hybrid")

def build_densenet121():
    base = DenseNet121(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = FINE_TUNE_BASE
    inp = layers.Input(shape=IMG_SIZE + (3,))
    x = _maybe_augment(inp)
    x = layers.Rescaling(1.0 / 255)(x)
    x = base(x)
    x = layers.Conv2D(DENSENET_CONV_FILTERS, 3, padding="same", activation="relu")(x)
    x = layers.GlobalMaxPooling2D()(x)
    x = layers.Dense(DENSENET_FC1_UNITS, activation="relu")(x)
    x = layers.Dense(DENSENET_FC2_UNITS, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return models.Model(inp, out, name="DenseNet121_hybrid")

def build_convnext_large():
    # Keras ConvNeXt normalizes internally and expects RAW 0-255 -> no Rescaling here.
    base = ConvNeXtLarge(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = CONVNEXT_FINE_TUNE_BASE
    inp = layers.Input(shape=IMG_SIZE + (3,))
    x = _maybe_augment(inp)
    x = base(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(CONVNEXT_FC_UNITS, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return models.Model(inp, out, name="ConvNeXtLarge_hybrid")

BUILDERS = {"inceptionv3":  build_inceptionv3,
            "densenet121":  build_densenet121,
            "convnextlarge": build_convnext_large}
print("Builders ready for:", BACKBONES)


In [ ]:
# ---------------------------------------------------------------------------
# B5. Resume-safe training. One optimizer, LR, epoch count for every backbone
#     and every arm (LIMITATION 8).
# ---------------------------------------------------------------------------
def train_resume_safe(name):
    ckpt_dir = os.path.join(CHECKPOINT_DIR, name)
    os.makedirs(ckpt_dir, exist_ok=True)
    last_path = os.path.join(ckpt_dir, "last.keras")
    best_path = os.path.join(ckpt_dir, "best.keras")
    log_path  = os.path.join(ckpt_dir, "history.csv")

    initial_epoch = 0
    if os.path.exists(last_path):
        model = tf.keras.models.load_model(last_path)
        if os.path.exists(log_path):
            initial_epoch = int(pd.read_csv(log_path)["epoch"].max()) + 1
        print(f"[{name}] resuming from epoch {initial_epoch}")
    else:
        print(f"[{name}] training from scratch")
        tf.keras.utils.set_random_seed(SEED)   # identical head init every run
        model = BUILDERS[name]()
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
                      loss="sparse_categorical_crossentropy",
                      metrics=["accuracy"])

    if initial_epoch >= EPOCHS:
        print(f"[{name}] already at {EPOCHS} epochs")
        return model, pd.read_csv(log_path)

    cbs = [tf.keras.callbacks.ModelCheckpoint(last_path, save_freq="epoch"),
           tf.keras.callbacks.ModelCheckpoint(best_path, monitor="val_accuracy",
                                              save_best_only=True),
           tf.keras.callbacks.CSVLogger(log_path, append=True)]
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
              initial_epoch=initial_epoch, callbacks=cbs,
              class_weight=CLASS_WEIGHT, verbose=1)
    return model, pd.read_csv(log_path)


In [ ]:
# ---------------------------------------------------------------------------
# B6. Train every backbone in this arm.
#     Safe to re-run after a Colab disconnect -- finished backbones are skipped
#     and the interrupted one resumes from its last epoch.
# ---------------------------------------------------------------------------
trained, histories = {}, {}
for name in BACKBONES:
    print("\n" + "=" * 70)
    print(f"TRAINING {name}   (arm={ARM}, seed={SEED})")
    print("=" * 70)
    trained[name], histories[name] = train_resume_safe(name)


In [ ]:
# ---------------------------------------------------------------------------
# B7. Predict on Derm7pt and save the per-model probabilities.
#     These .npy files are the ONLY thing Part A needs.
# ---------------------------------------------------------------------------
y_true_run = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
np.save(os.path.join(RESULTS_DIR, "y_true.npy"), y_true_run)

FILE_FOR = {"inceptionv3": "prob_inception.npy",
            "densenet121": "prob_densenet.npy",
            "convnextlarge": "prob_convnext.npy"}

for name in BACKBONES:
    p = trained[name].predict(test_ds, verbose=1)
    np.save(os.path.join(RESULTS_DIR, FILE_FOR[name]), p)
    print(f"{name}: saved {FILE_FOR[name]}  shape={p.shape}")

# --- manifest: the audit trail a reviewer can check ------------------------
manifest = {
    "arm": ARM, "seed": SEED, "backbones": BACKBONES,
    "batch_size": ACTIVE_BATCH, "learning_rate": LR, "epochs": EPOCHS,
    "val_split": VAL_SPLIT, "optimizer": "Adam",
    "loss": "sparse_categorical_crossentropy",
    "augmentation": {"enabled": bool(ACTIVE_AUG), "hflip": AUG_HFLIP,
                     "rotation": AUG_ROTATION, "noise_stddev": NOISE_STDDEV,
                     "noise_on_255_scale": NOISE_ON_255_SCALE},
    "fine_tune": {"inception_densenet": FINE_TUNE_BASE,
                  "convnext": CONVNEXT_FINE_TUNE_BASE},
    "class_weights": CLASS_WEIGHT,
    "accurate_decode": ACCURATE_DECODE,
    "deterministic": DETERMINISTIC,
    "n_train": int(len(tr_df)), "n_val": int(len(va_df)), "n_test": int(len(test_df)),
    "train_split_fingerprint": split_hash,
    "test_order_fingerprint": test_hash,
    "tensorflow": tf.__version__, "keras": tf.keras.__version__,
}
with open(os.path.join(RESULTS_DIR, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)
print("\nManifest ->", os.path.join(RESULTS_DIR, "manifest.json"))
print(json.dumps(manifest, indent=2))
print("\nDone. Set MODE='analyze' and run Part A once all arms/seeds exist.")


---
## What to write in the paper

Once Part A has run, the ablation section writes itself. A defensible paragraph:

> To isolate the contribution of the third backbone, we conducted a controlled ablation
> in which H2 (InceptionV3 + DenseNet121) and H3 (InceptionV3 + DenseNet121 +
> ConvNeXt-Large) share the identical training harness: batch size (16), augmentation
> (horizontal flip and additive Gaussian noise), optimizer (Adam), learning rate
> (1e-4), epoch budget (50), and the same stratified 90/10 HAM10000 train/validation
> split under seeds {42, 1000, 50000}. The two configurations are evaluated from the
> *same* trained InceptionV3 and DenseNet121 weights; H3 differs only by the addition of
> the ConvNeXt-Large probability stream to the weighted-sum fusion, with the two shared
> weights renormalised (0.455 / 0.545) so the InceptionV3-to-DenseNet121 ratio is
> preserved. Table&nbsp;V therefore attributes the observed change to the third backbone
> alone. We additionally report H2 under its original harness (batch 32, no augmentation)
> to separate the harness effect from the architectural effect. Statistical significance
> was assessed with an exact McNemar test on the paired per-image predictions and a
> paired bootstrap (2000 resamples) of the macro-F1 difference.

### Checklist before submission

| # | Limitation | Where it is addressed |
|---|---|---|
| 5 | Controlled H2 vs H3 ablation | Part A4 &mdash; both arms from one set of weights |
| 6 | Identical batch size | `BATCH_SIZE = 16`, shared; recorded in `manifest.json` |
| 7 | Identical augmentation | one `augmentation_block()` used by every backbone in the arm |
| 8 | Identical optimizer / LR / epochs / split | single CONFIG; split fingerprint in the manifest |
| 9 | Clean definition of H2 and H3 | `ARM_SPECS` in A4; weights renormalised, ratio preserved |
| 10 | Three-row ablation table | `ablation/ablation_table.csv` and `.tex` |
| 11 | Figures traceable to Table III runs | A6 &mdash; figures built from the reported seed's saved probabilities; `figure_provenance.json` |

Everything a reviewer might ask for lands in `BASE_DIR/ablation/`:
`ablation_per_seed.csv`, `ablation_table.csv`, `ablation_mean_sd.csv`,
`significance_h2c_vs_h3c.csv`, `ablation_table.tex`, `figure_provenance.json`,
plus the seed-stamped confusion matrix, ROC and per-class table.
